# Task
Process the `cleaned_energy_consumption_data.xlsx` dataset by performing time-based feature engineering, creating lag and rolling features, separating data by appliance, and then splitting each appliance's data into training and testing sets for time-series forecasting, saving the resulting datasets into separate CSV files.

## Load and Inspect Data

### Subtask:
Load the hourly device-level energy consumption data, parse the timestamp column as datetime objects, and confirm the number of records, appliance types, and initial absence of missing values as described.


**Reasoning**:
Load the Excel file into a pandas DataFrame, parsing the timestamp column as datetime objects, and then perform initial data inspections as requested.



In [3]:
import pandas as pd

# 1. Load the Excel file into a pandas DataFrame and parse the 'Timestamp' column (assuming case sensitivity was the issue)
df_energy = pd.read_excel('/content/cleaned_energy_consumption_data.xlsx', parse_dates=['Timestamp'])

# 2. Print the shape of the DataFrame
print(f"DataFrame shape: {df_energy.shape}")

# 3. Identify and print the unique appliance types
unique_appliances = df_energy['Appliance Type'].unique()
print(f"Unique Appliance types: {unique_appliances}")

# 4. Check for any missing values across all columns and print a summary
missing_values = df_energy.isnull().sum()
print("\nMissing values per column:\n", missing_values[missing_values > 0])

DataFrame shape: (100000, 10)
Unique Appliance types: ['Air Conditioning' 'Computer' 'Dishwasher' 'Fridge' 'Heater' 'Lights'
 'Microwave' 'Oven' 'TV' 'Washing Machine']

Missing values per column:
 Series([], dtype: int64)


## Time-Based Feature Engineering

### Subtask:
From the parsed timestamp, create new features: 'hour' (0-23), 'day' (day of the month), 'weekday' (0=Monday), and 'month' (month of the year) to capture temporal patterns.


**Reasoning**:
I will create new features ('hour', 'day', 'weekday', 'month') from the 'Timestamp' column to capture temporal patterns, which is a common practice in time series analysis to provide more granular information to a model.



In [4]:
df_energy['hour'] = df_energy['Timestamp'].dt.hour
df_energy['day'] = df_energy['Timestamp'].dt.day
df_energy['weekday'] = df_energy['Timestamp'].dt.weekday
df_energy['month'] = df_energy['Timestamp'].dt.month

# Display the first few rows with the new features
print(df_energy[['Timestamp', 'hour', 'day', 'weekday', 'month']].head())

            Timestamp  hour  day  weekday  month
0 2023-07-01 23:32:00    23    1        5      7
1 2023-08-24 01:43:00     1   24        3      8
2 2023-08-24 01:36:00     1   24        3      8
3 2023-05-06 10:47:00    10    6        5      5
4 2023-01-29 22:39:00    22   29        6      1


## Create Lag and Rolling Features

### Subtask:
Generate 'lag_1' (energy consumption 1 hour ago), 'lag_24' (energy consumption 24 hours ago), and 'rolling_mean_24' (average consumption over the past 24 hours). Subsequently, remove rows with missing lag values using `dropna()`.


**Reasoning**:
I will sort the DataFrame, then group by 'Appliance Type' to calculate 'lag_1', 'lag_24', and 'rolling_mean_24' for 'Energy Consumption (kWh)', and finally drop rows with NaN values introduced by these operations, as specified in the instructions.



In [5]:
import numpy as np

# 1. Sort the DataFrame by 'Appliance Type' and 'Timestamp'
df_energy = df_energy.sort_values(by=['Appliance Type', 'Timestamp'])

# 2. For each 'Appliance Type', create the 'lag_1' feature
df_energy['lag_1'] = df_energy.groupby('Appliance Type')['Energy Consumption (kWh)'].shift(1)

# 3. For each 'Appliance Type', create the 'lag_24' feature
df_energy['lag_24'] = df_energy.groupby('Appliance Type')['Energy Consumption (kWh)'].shift(24)

# 4. For each 'Appliance Type', calculate the 'rolling_mean_24' feature
# Calculate rolling mean and then shift by 1 to represent the average of the *previous* 24 hours.
df_energy['rolling_mean_24'] = df_energy.groupby('Appliance Type')['Energy Consumption (kWh)'] \
                                    .transform(lambda x: x.rolling(window=24, min_periods=1).mean().shift(1))

# 5. Remove any rows that contain missing values introduced by the lag and rolling calculations
df_energy.dropna(inplace=True)

# 6. Display the first few rows of the DataFrame, including the newly created features
print(df_energy[['Appliance Type', 'Timestamp', 'Energy Consumption (kWh)', 'lag_1', 'lag_24', 'rolling_mean_24']].head())

        Appliance Type           Timestamp  Energy Consumption (kWh)  lag_1  \
7585  Air Conditioning 2023-01-01 17:11:00                      2.81   2.89   
7581  Air Conditioning 2023-01-01 18:40:00                      3.91   2.81   
7580  Air Conditioning 2023-01-01 18:43:00                      3.66   3.91   
7563  Air Conditioning 2023-01-01 19:31:00                      2.74   3.66   
7591  Air Conditioning 2023-01-01 21:50:00                      3.49   2.74   

      lag_24  rolling_mean_24  
7585    4.42         3.620833  
7581    4.37         3.553750  
7580    3.24         3.534583  
7563    3.75         3.552083  
7591    4.22         3.510000  


## Appliance-Wise Data Separation

### Subtask:
Identify the 10 unique appliance types. For each appliance, filter the data to be appliance-specific, then separate the features (X), which include time and lag variables, from the target (y), which is Energy Consumption (kWh).


**Reasoning**:
I will iterate through each unique appliance, filter the main DataFrame, and then separate the features (X) from the target (y) for each appliance, storing them in a dictionary as specified.



In [6]:
appliance_data = {}

for appliance in unique_appliances:
    # Filter data for the current appliance
    df_appliance = df_energy[df_energy['Appliance Type'] == appliance].copy()

    # Define target variable (y)
    y_appliance = df_appliance['Energy Consumption (kWh)']

    # Define feature variables (X)
    X_appliance = df_appliance[[
        'hour', 'day', 'weekday', 'month', 'Outdoor Temperature',
        'Household Size', 'lag_1', 'lag_24', 'rolling_mean_24'
    ]]

    # Store X and y for the current appliance
    appliance_data[appliance] = {'X': X_appliance, 'y': y_appliance}

# Print the keys of the dictionary to confirm separation
print("Appliance types for which data has been separated:", appliance_data.keys())

Appliance types for which data has been separated: dict_keys(['Air Conditioning', 'Computer', 'Dishwasher', 'Fridge', 'Heater', 'Lights', 'Microwave', 'Oven', 'TV', 'Washing Machine'])


## Perform Time-Series Train-Test Split

### Subtask:
For each appliance's separated data, perform a time-series safe split, using 80% of the earliest data for training and the remaining 20% of the latest data for testing, without any random shuffling.


**Reasoning**:
I need to initialize dictionaries to store the split data, then iterate through each appliance's data to perform a time-series safe 80/20 train-test split, and finally store the results in the respective dictionaries.



In [7]:
X_train_data = {}
X_test_data = {}
y_train_data = {}
y_test_data = {}

for appliance, data in appliance_data.items():
    X = data['X']
    y = data['y']

    # Calculate the split point (80% for training)
    split_point = int(len(X) * 0.8)

    # Split data for training and testing
    X_train = X.iloc[:split_point]
    X_test = X.iloc[split_point:]
    y_train = y.iloc[:split_point]
    y_test = y.iloc[split_point:]

    # Store the split data in the dictionaries
    X_train_data[appliance] = X_train
    X_test_data[appliance] = X_test
    y_train_data[appliance] = y_train
    y_test_data[appliance] = y_test

# Confirm that data for all appliances has been split and stored
print("Appliance types for which X_train data has been stored:", X_train_data.keys())
print("Number of training samples for 'Air Conditioning':", len(X_train_data['Air Conditioning']))
print("Number of testing samples for 'Air Conditioning':", len(X_test_data['Air Conditioning']))

Appliance types for which X_train data has been stored: dict_keys(['Air Conditioning', 'Computer', 'Dishwasher', 'Fridge', 'Heater', 'Lights', 'Microwave', 'Oven', 'TV', 'Washing Machine'])
Number of training samples for 'Air Conditioning': 8034
Number of testing samples for 'Air Conditioning': 2009


## Save Processed Datasets

### Subtask:
Save the processed datasets for each appliance into separate CSV files: X_train_<appliance>.csv, X_test_<appliance>.csv, y_train_<appliance>.csv, and y_test_<appliance>.csv.


**Reasoning**:
I need to save the training and testing datasets for each appliance into separate CSV files, following the specified naming convention and ensuring no index is written to the file. I will iterate through the stored appliance data and use the `to_csv` method.



In [8]:
import os

# Create a directory to store the processed datasets if it doesn't exist
output_dir = '/content/processed_appliance_data'
os.makedirs(output_dir, exist_ok=True)

saved_appliances = []

for appliance in X_train_data.keys():
    # Save X_train data
    X_train_data[appliance].to_csv(f"{output_dir}/X_train_{appliance.replace(' ', '_')}.csv", index=False)

    # Save X_test data
    X_test_data[appliance].to_csv(f"{output_dir}/X_test_{appliance.replace(' ', '_')}.csv", index=False)

    # Save y_train data
    y_train_data[appliance].to_csv(f"{output_dir}/y_train_{appliance.replace(' ', '_')}.csv", index=False, header=True)

    # Save y_test data
    y_test_data[appliance].to_csv(f"{output_dir}/y_test_{appliance.replace(' ', '_')}.csv", index=False, header=True)

    saved_appliances.append(appliance)

print(f"Successfully saved processed datasets for the following appliances: {', '.join(saved_appliances)}")

Successfully saved processed datasets for the following appliances: Air Conditioning, Computer, Dishwasher, Fridge, Heater, Lights, Microwave, Oven, TV, Washing Machine


## Final Task

### Subtask:
Summarize the completion of the data processing and preparation steps, confirming that all specified datasets have been generated and saved for each appliance.


## Summary:

### Q&A
All specified datasets (X\_train, X\_test, y\_train, and y\_test) have been successfully generated and saved for each of the 10 unique appliance types: 'Air Conditioning', 'Computer', 'Dishwasher', 'Fridge', 'Heater', 'Lights', 'Microwave', 'Oven', 'TV', and 'Washing Machine'. These datasets are stored in separate CSV files within the `/content/processed_appliance_data` directory.

### Data Analysis Key Findings
*   The initial dataset, `cleaned_energy_consumption_data.xlsx`, containing 100,000 records and 10 columns, was successfully loaded. It included 10 unique appliance types and had no missing values.
*   Four new time-based features ('hour', 'day', 'weekday', 'month') were successfully extracted from the 'Timestamp' column.
*   Lag features ('lag\_1', 'lag\_24') and a rolling mean feature ('rolling\_mean\_24') were computed for 'Energy Consumption (kWh)', grouped by 'Appliance Type'. Rows with missing values introduced by these calculations were removed.
*   The data was successfully separated for each of the 10 unique appliance types. For each appliance, features (X) included 'hour', 'day', 'weekday', 'month', 'Outdoor Temperature', 'Household Size', 'lag\_1', 'lag\_24', and 'rolling\_mean\_24', while the target (y) was 'Energy Consumption (kWh)'.
*   A time-series safe split was performed for each appliance's data, allocating 80% for training and 20% for testing. For instance, the 'Air Conditioning' dataset was split into 8034 training samples and 2009 testing samples.
*   All processed datasets (X\_train, X\_test, y\_train, y\_test) for each appliance were saved into individual CSV files within the `/content/processed_appliance_data` directory, following a consistent naming convention (e.g., `X_train_Air_Conditioning.csv`).

### Insights or Next Steps
*   The structured and prepared datasets are now ready for training and evaluating time-series forecasting models for each specific appliance.
*   Further analysis can explore the individual performance of forecasting models across different appliance types to identify which features or models are most effective for specific consumption patterns.
